In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import ast
import re
from scipy.stats import mannwhitneyu

In [2]:
events_df = pd.read_csv('../../SV_signature_analysis/Clustered_SVs/results/hotspot_event_definitions.csv')
events_df['chrom'] = 'chr' + events_df['chrom'].astype(str)
hotspots = (
    events_df
    .set_index('event')[['chrom', 'start', 'end']]
    .to_dict(orient='index')
)
hotspots

{'CCND3': {'chrom': 'chr6', 'start': 39000000, 'end': 55000000},
 'MYC': {'chrom': 'chr8', 'start': 99000000, 'end': 137000000},
 'CDK4-MDM2': {'chrom': 'chr12', 'start': 53000000, 'end': 75000000},
 'CCNE1': {'chrom': 'chr19', 'start': 27000000, 'end': 34000000},
 'TP53': {'chrom': 'chr17', 'start': 7000000, 'end': 22000000}}

In [4]:
chrom_sizes = pd.read_csv('../data/hg38_chr.bed', sep='\t', header=None)
chrom_sizes.columns = ['chrom', 'start', 'end', 'name']
chrom_sizes = chrom_sizes.set_index('chrom')['end'].to_dict()
# remove chrY
chrom_sizes.pop('chrY', None)
chrom_sizes.pop('#chrom', None)
# make sure sizes are ints
chrom_sizes = {k: int(v) for k, v in chrom_sizes.items()}
chrom_sizes

{'chr1': 248956422,
 'chr2': 242193529,
 'chr3': 198295559,
 'chr4': 190214555,
 'chr5': 181538259,
 'chr6': 170805979,
 'chr7': 159345973,
 'chr8': 145138636,
 'chr9': 138394717,
 'chr10': 133797422,
 'chr11': 135086622,
 'chr12': 133275309,
 'chr13': 114364328,
 'chr14': 107043718,
 'chr15': 101991189,
 'chr16': 90338345,
 'chr17': 83257441,
 'chr18': 80373285,
 'chr19': 58617616,
 'chr20': 64444167,
 'chr21': 46709983,
 'chr22': 50818468,
 'chrX': 156040895}

In [5]:
bins_df = pd.read_csv('../results/ecdna_windows.csv') # 100kb bins

# label each bin as hotspot or background
def in_hotspot(row):
    for locus, coords in hotspots.items():
        if (row['chrom'] == coords['chrom'] and
                row['window_start_bp'] < coords['end'] and
                row['window_end_bp']   > coords['start']):
            return locus
    return None

bin_size = 100_000
all_bins = []
for chrom, size in chrom_sizes.items():
    starts = range(0, size, bin_size)
    for s in starts:
        all_bins.append({
            'chrom': chrom,
            'window_start_bp': s,
            'window_end_bp': min(s + bin_size, size)
        })

all_bins_df = pd.DataFrame(all_bins)

# merge with observed counts, fill missing with 0
bins_full = all_bins_df.merge(
    bins_df[['chrom', 'window_start_bp', 'n_samples']],
    on=['chrom', 'window_start_bp'],
    how='left'
).fillna({'n_samples': 0})

bins_full['n_samples'] = bins_full['n_samples'].astype(int)

# label hotspots
bins_full['hotspot'] = bins_full.apply(in_hotspot, axis=1)

hotspot_bins    = bins_full[bins_full['hotspot'].notna()]['n_samples']
background_bins = bins_full[bins_full['hotspot'].isna()]['n_samples']

stat, p = mannwhitneyu(hotspot_bins, background_bins, alternative='greater')
fold    = hotspot_bins.mean() / background_bins.mean()

print(f"Hotspot bins     : {len(hotspot_bins)}, mean = {hotspot_bins.mean():.3f}")
print(f"Background bins  : {len(background_bins)}, mean = {background_bins.mean():.4f}")
print(f"Fold enrichment  : {fold:.1f}x")
print(f"p-value          : {p:.2e}")

Hotspot bins     : 980, mean = 0.493
Background bins  : 29341, mean = 0.0228
Fold enrichment  : 21.6x
p-value          : 5.39e-69
